# Newspaper Article Extraction from Scanned PDFs

This notebook extracts individual articles from scanned newspaper PDFs (tested on: Shanghai Jewish Chronicle, Internet Archive scan) and prepares them for a subsequent sentiment analysis using the NRC Emotion Lexicon.

## Pipeline overview:

    1. PDF page -> render as high-resolution image (PyMuPDF)
    2. Custom OCR with Tesseract (German and English combined)
    3. Group words -> lines -> articles (font size as headline signal)
    4. Filter out advertisements (box numbers, phone numbers, ...)
    5. Extract date/year from the newspaper masthead
    6. Detect language per article (German vs. English -- the paper's own masthead states "English and German Editions")
    7. Helper functions for the later NRC lexicon (umlaut substitution such as "fuer" instead of "fur", historically standard in letterpress typesetting)
    8. Main pipeline: a single page / a whole PDF / a whole folder of PDFs, optionally parallelized across multiple CPU cores
    9. Sample validation

## Important note:
The thresholds (headline detection, advertisement filter) were calibrated on a test page. Before interpreting real results, always use Section 9 (sample validation) -- ideally separately for different years, since the newspaper's layout changed over time (see comments below).


# 0. Setup

Required packages:

    pymupdf -- render PDF pages as images
    pandas -- data handling/CSV
    langdetect -- per-article language detection
    the Tesseract CLI (tesseract) with the language models deu.traineddata and eng.traineddata installed

If the language models are missing, see the commented-out download commands below

In [ ]:


# If not yet installed:
!pip install pymupdf pandas langdetect --quiet
import pymupdf

# If the Tesseract language models are missing (example for Linux):
# !curl -sL -o deu.traineddata https://raw.githubusercontent.com/tesseract-ocr/tessdata_fast/main/deu.traineddata
# !curl -sL -o eng.traineddata https://raw.githubusercontent.com/tesseract-ocr/tessdata_fast/main/eng.traineddata
# Then copy both files into the Tesseract tessdata directory
# (e.g. /usr/share/tesseract-ocr/5/tessdata/ -- check the path with `tesseract --list-langs`
# or the error message from Tesseract).





In [ ]:
import logging
import os
import re
import subprocess
import tempfile
import time
from concurrent.futures import ThreadPoolExecutor, TimeoutError
from pathlib import Path

import pymupdf
import fitz  # PyMuPDF
import pandas as pd
from langdetect import detect, DetectorFactory

DetectorFactory.seed = 0  # makes langdetect deterministic (otherwise random)

# Logging: writes progress both to a file (extraction.log) and to the
# notebook output -- important for longer runs, so that afterwards one
# can trace how long each page took or where an error occurred.
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler("extraction.log", encoding="utf-8"),
        logging.StreamHandler(),
    ],
    force=True,  # overrides any logging configuration already in place
)
log = logging.getLogger(__name__)

# 1. Configuration

All thresholds and patterns in one place, so they can be easily adjusted (e.g. if a later year turns out to be set more densely and needs different values).

In [ ]:
# --- OCR settings -------------------------------------------------
DPI = 200                    # Trade-off between OCR quality and runtime.
                              # Higher DPI barely improves quality further,
                              # but substantially increases runtime.
LANG = "deu+eng"              # Combined language model: the paper's own
                              # masthead states "English and German
                              # Editions" -- about 11% of articles in the
                              # test corpus are actually English.
MIN_WORD_CONF = 30           # Tesseract confidence threshold (0-100) per word

# --- Article segmentation ----------------------------------------------
HEADLINE_FACTOR = 1.6        # A line counts as a headline (= start of a
                              # new article) if its height > median line
                              # height * factor.
MIN_ARTICLE_WORDS = 40       # Shorter text blocks are discarded
                              # (isolated headlines, captions, ...).

# OCR noise filter at word level: single punctuation marks, extremely
# long "words" (lines merged without spaces), etc. These distort both
# the line-height statistics (headline detection) and later the
# NRC scoring, so they are removed here already.
MIN_WORD_LEN = 1
MAX_WORD_LEN = 40

# --- Advertisement detection --------------------------------------------------
# "Chiffre" (anonymous box-number contact code) occurs in this corpus
# almost exclusively in classified ads -- a very reliable signal.
AD_PATTERNS = [
    r"\bChiffre\b",
    r"\bTel\.?\s?\d{4,6}\b",
    r"\bPhone\s?\d{4,6}\b",
    r"\b(Road|Rd\.?|Strasse|Str\.?)\s+\d{1,4}\b",
    r"\bPreis(e)?\s?$",
    r"\bzu\s+verkaufen\b",
    r"\bgesucht\b.{0,15}\bChiffre\b",
]
AD_REGEX = re.compile("|".join(AD_PATTERNS), re.IGNORECASE)

# Press-agency abbreviations that typically appear directly below an
# article headline -- not an ad filter, but a quality signal ("this is
# very likely a genuine news article").
AGENCY_MARKERS = [
    "Reuter", "Transocean", "Havas", "D.N.B.", "DNB", "CPS", "CPS-Berlin",
    "Central Press",
]

# --- Date detection -----------------------------------------------------
# Regex for the publication date in the newspaper masthead, e.g.
# "Shanghai Donnerstag 22. Juni 1939". Tolerates common OCR errors such
# as "i939" instead of "1939" (the leading "1" is often read as "i"/"l").
WEEKDAYS = r"(Montag|Dienstag|Mittwoch|Donnerstag|Freitag|Samstag|Sonnabend|Sonntag)"
MONTHS = r"(Januar|Februar|Maerz|M[\u00e4a]rz|April|Mai|Juni|Juli|August|September|Oktober|November|Dezember)"
DATE_PATTERN = re.compile(
    rf"{WEEKDAYS}\.?,?\s+(\d{{1,2}})\.?\s*{MONTHS}\.?,?\s*[il]?(\d{{3,4}})",
    re.IGNORECASE,
)
MONTH_NUM = {
    "januar": 1, "februar": 2, "maerz": 3, "m\u00e4rz": 3, "april": 4, "mai": 5,
    "juni": 6, "juli": 7, "august": 8, "september": 9, "oktober": 10,
    "november": 11, "dezember": 12,
}

print("Configuration loaded.")


# 2. Step 1+2: Render PDF page and run OCR

Instead of using the OCR text layer embedded in the PDF (in tests: scrambled reading order, many misread characters), each page is rendered as a high-resolution image and Tesseract is run on it directly.

The trick: Tesseract's TSV output provides not just the raw text but also layout information per word -- block_num/par_num/line_num (Tesseract's own column/paragraph detection) as well as the bounding box and a confidence score. This information is needed next to determine line heights (= font size) and thereby detect headlines.

In [ ]:
def render_page(doc, page_num, dpi=DPI, out_path=None):
    """Renders a PDF page as a PNG."""
    page = doc[page_num]
    pix = page.get_pixmap(dpi=dpi)
    out_path = out_path or tempfile.mktemp(suffix=".png")
    pix.save(out_path)
    return out_path


def ocr_page_tsv(image_path, lang=LANG):
    """Calls Tesseract and returns the word level as a DataFrame.

    The TSV output contains per word: block_num/par_num/line_num, the
    bounding box (left/top/width/height), and a confidence score (conf, 0-100).
    """
    out_base = tempfile.mktemp()
    try:
        subprocess.run(
            ["tesseract", image_path, out_base, "-l", lang, "--psm", "3", "tsv"],
            check=True, capture_output=True, timeout=120,
        )
        df = pd.read_csv(out_base + ".tsv", sep="\t", quoting=3)
    finally:
        tsv_path = out_base + ".tsv"
        if os.path.exists(tsv_path):
            os.remove(tsv_path)

    df = df[df["level"] == 5]  # word level only (level 5 = single word)
    df = df[df["conf"].astype(float) >= MIN_WORD_CONF]
    df = df[df["text"].notna()]
    df = df[df["text"].astype(str).str.len().between(MIN_WORD_LEN, MAX_WORD_LEN)]
    return df

print("OCR functions defined.")

# 3. Step: group words -> lines -> articles

Three stages:
    1. Individual OCR words are grouped into lines (grouped by Tesseract's block_num/par_num/line_num).
    2. Lines are grouped into articles: a noticeably large line (headline) starts a new article.
    3. Minor text cleanup (multiple spaces, hyphenation).

In [ ]:
def words_to_lines(word_df):
    """Groups individual OCR words into lines."""
    if word_df.empty:
        return pd.DataFrame(columns=["block_num", "par_num", "line_num",
                                      "text", "height", "top", "left"])
    lines = (
        word_df.sort_values(["block_num", "par_num", "line_num", "word_num"])
        .groupby(["block_num", "par_num", "line_num"])
        .agg(
            text=("text", lambda x: " ".join(str(w) for w in x)),
            height=("height", "mean"),
            top=("top", "min"),
            left=("left", "min"),
        )
        .reset_index()
        .sort_values(["block_num", "par_num", "top"])
    )
    return lines


def group_into_articles(lines, headline_factor=HEADLINE_FACTOR,
                         min_words=MIN_ARTICLE_WORDS):
    """Groups lines into articles. A noticeably large new line
    (headline) starts a new article."""
    if lines.empty:
        return []

    median_height = lines["height"].median()
    headline_threshold = median_height * headline_factor

    articles = []
    current_lines = []

    for _, row in lines.iterrows():
        is_headline = row["height"] >= headline_threshold and len(row["text"].split()) <= 12
        if is_headline and current_lines:
            articles.append(_finalize_article(current_lines))
            current_lines = [row]
        else:
            current_lines.append(row)
    if current_lines:
        articles.append(_finalize_article(current_lines))

    # discard "articles" that are too short (e.g. isolated headlines without body text)
    articles = [a for a in articles if len(a["text"].split()) >= min_words]
    return articles


def _finalize_article(rows):
    text = " ".join(r["text"] for r in rows) if isinstance(rows, list) else \
        " ".join(rows["text"])
    top = min(r["top"] for r in rows) if isinstance(rows, list) else rows["top"].min()
    return {"text": text.strip(), "top": top}


def clean_text(text):
    """Minor text cleanup: removes multiple spaces and merges simple
    end-of-line hyphenation cases (e.g. "Kran- kheit" -> "Krankheit").
    A heuristic, not always correct, but noticeably improves the
    later NRC lexicon match rate."""
    text = re.sub(r"(\w)-\s+(\w)", r"\1\2", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

print("Segmentation functions defined.")

# 4. Step: filter out advertisements

A combination of several signals: text too short, typical advertisement patterns ("Chiffre", phone numbers, address patterns), high digit ratio.

In [ ]:
def is_advertisement(text, min_words=MIN_ARTICLE_WORDS):
    word_count = len(text.split())
    if word_count < min_words:
        return True
    if AD_REGEX.search(text):
        return True
    digit_ratio = sum(c.isdigit() for c in text) / max(len(text), 1)
    if digit_ratio > 0.12:
        return True
    return False


def has_agency_marker(text):
    return any(marker.lower() in text.lower() for marker in AGENCY_MARKERS)

print("Advertisement filter defined.")

# 5. Step: extract date/year from the newspaper masthead

The date appears in the masthead of every issue (e.g. "Shanghai Donnerstag 22. Juni 1939"). The regex tolerates typical OCR errors in the year -- the leading "1" is often read as "i"/"l" and lost ("1939" -> "i939" -> digit group only "939"). This is detected here and corrected to four digits.

In [ ]:
def extract_date(text):
    m = DATE_PATTERN.search(text)
    if not m:
        return None, None
    _, day, month, year = m.groups()
    month_num = MONTH_NUM.get(month.lower().replace("\u00e4", "ae"))
    year = int(year)
    # OCR sometimes drops the leading "1" of a year
    # ("i939" -> digit group only "939"). Pad back to four digits.
    if 100 <= year < 1000:
        year += 1000
    elif year < 100:
        year += 1900
    return year, month_num


# Quick test:
print(extract_date("Shanghai Donnerstag 22 Juni. i939"))  # should give (1939, 6)

# 6. Step: language per article + NRC lexicon helper functions

Two things that matter for the later sentiment analysis:

a) Language per article. The paper's masthead states "English and German Editions" -- in the test corpus, about 11% of articles were actually English. For NRC scoring, the matching lexicon (German or English) must be used per article, otherwise English words are matched against the German lexicon (or vice versa) and produce meaningless scores near zero.

b) Umlaut substitution. Historical newspaper typesetting consistently uses ae/oe/ue/ss instead of ä/ö/ü/ß (no umlaut character was available in the letterpress/telegraphy character set of the time) -- this is not an OCR error but how the text was actually printed. In the test corpus, "fuer" alone occurs 229 times. The transliteration is deliberately applied to the NRC lexicon ("über" -> "ueber"), not the other way around on the corpus text: converting "ue" in the text back to "ü" would be ambiguous (words like "Museum" or "neu" contain "ue"/"eu" without an umlaut being intended).

In [ ]:
def detect_language(text):
    """Detects the language of an article (de/en/...). Returns "unknown"
    for very short/ambiguous texts instead of guessing."""
    if len(text.split()) < 5:
        return "unknown"
    try:
        return detect(text)
    except Exception:
        return "unknown"


# Substitution as used in historical typesetting: no umlaut/sharp S
# available in the character set, hence "ae/oe/ue/ss" instead of "ä/ö/ü/ß".
_UMLAUT_MAP = str.maketrans({
    "\u00e4": "ae", "\u00f6": "oe", "\u00fc": "ue",
    "\u00c4": "Ae", "\u00d6": "Oe", "\u00dc": "Ue",
})


def delatinize(word):
    """Transliterates a word with 'real' umlauts/sharp S into the
    substitution spelling used in the corpus, e.g. 'ueber' <- 'über',
    'fuer' <- 'für', 'moechte' <- 'möchte', 'Strasse' <- 'Straße'."""
    word = word.replace("\u00df", "ss")
    return word.translate(_UMLAUT_MAP)


def delatinize_lexicon(word2emotions):
    """Extends a loaded NRC lexicon (word -> set of emotions) with
    transliterated additional entries, so that 'fuer', 'ueber', 'moechte'
    are also matched in the corpus. Existing entries are preserved;
    on collisions, the emotion sets are merged."""
    extended = dict(word2emotions)
    for word, emotions in word2emotions.items():
        alt = delatinize(word)
        if alt != word:
            extended[alt] = extended.get(alt, set()) | emotions
    return extended


# Quick test:
for w in ["f\u00fcr", "\u00fcber", "k\u00f6nnen", "m\u00f6chte"]:
    print(w, "->", delatinize(w))

# 7. Main pipeline: a single page / a whole PDF / a whole folder

Three functions, building on each other:

    process_page() -- processes exactly one page.
    process_pdf() -- processes a whole PDF, serially or in parallel via a thread pool, with running checkpointing and a hard timeout per page.
    process_folder() -- processes all PDFs in a folder (e.g. one file per year), including a source_file column.

Practical lesson: on a real 227-page run, an earlier version of this script appeared to "hang" for 14 hours. The likely causes were (a) memory thrashing -- one page at 300 DPI uses roughly 250 MB of RAM uncompressed, and with too many parallel workers the available memory is exceeded, the operating system starts swapping, and everything becomes 50-100x slower instead of faster; and (b) a design flaw: the original version used pool.starmap(), which only returns any result after all pages are done -- if a single page hangs, this is invisible, and the whole run appears to stall for no apparent reason.

This version addresses both issues:

    PAGE_TIMEOUT_SECONDS: a hard upper bound per page (rendering and OCR). If a page takes longer, it is skipped instead of blocking the run.
    Running checkpointing and progress logging, also in parallel mode: a log line with status and elapsed time every 5 pages -- no more hours of uncertainty about whether anything is still happening.
    Rule of thumb for n_workers: n_workers <= available RAM in GB / 1.5. Fewer workers are preferable to swapping.


In [ ]:
def process_page(pdf_path, page_num, n_pages, source_file=None):
    """Processes exactly one page, returns a list of article dicts."""
    source_file = source_file or os.path.basename(pdf_path)
    page_t0 = time.time()
    articles = []
    try:
        doc = fitz.open(pdf_path)  # reopen per process (a fitz.Document
                                    # cannot be safely shared across
                                    # process boundaries)
        img_path = render_page(doc, page_num)
        try:
            word_df = ocr_page_tsv(img_path)
        finally:
            if os.path.exists(img_path):
                os.remove(img_path)

        lines = words_to_lines(word_df)
        if lines.empty:
            log.info(f"{source_file} page {page_num + 1}/{n_pages}: no text detected.")
            return []

        page_text = " ".join(lines["text"])
        year, month = extract_date(page_text)

        candidates = group_into_articles(lines)
        for art in candidates:
            text = clean_text(art["text"])
            if is_advertisement(text):
                continue
            articles.append({
                "source_file": source_file,
                "pdf_page": page_num + 1,
                "year": year,
                "month": month,
                "lang": detect_language(text),
                "has_agency_marker": has_agency_marker(text),
                "word_count": len(text.split()),
                "text": text,
            })

        log.info(
            f"{source_file} page {page_num + 1}/{n_pages}: "
            f"{len(candidates)} blocks, {len(articles)} kept, "
            f"year={year}, {time.time() - page_t0:.1f}s"
        )
    except Exception as exc:
        log.error(f"{source_file} page {page_num + 1}/{n_pages}: ERROR -- {exc!r}")
        return []
    return articles




In [ ]:
PAGE_TIMEOUT_SECONDS = 180  # hard upper bound per page (rendering + OCR
                             # combined). Important: the Tesseract timeout
                             # in ocr_page_tsv() only protects the
                             # Tesseract subprocess, not the PDF
                             # rendering (fitz.get_pixmap()) before it --
                             # if that hangs or thrashes (e.g. due to
                             # memory pressure from too many parallel
                             # workers), there was previously no
                             # protection and no feedback at all.
                             # This is exactly what causes runs that
                             # appear to "hang" for hours in practice.


def process_pdf(pdf_path, page_range=None, n_workers=1, checkpoint_path=None):
    """Processes a whole PDF while preventing pool crashes.

    A thread pool is used here deliberately: notebook-defined functions
    from ``__main__`` cannot be reliably imported into spawned processes
    on macOS. The actual OCR work runs in Tesseract as an external
    process anyway, so threads do not block the notebook kernel.
    """
    doc = fitz.open(pdf_path)
    n_pages = len(doc)
    doc.close()
    source_file = os.path.basename(pdf_path)
    pages_to_process = list(range(n_pages) if page_range is None else page_range)

    all_articles = []
    t0 = time.time()
    executor = ThreadPoolExecutor(max_workers=n_workers)
    try:
        futures = {
            executor.submit(process_page, pdf_path, p, n_pages, source_file): p
            for p in pages_to_process
        }
        for i, future in enumerate(futures, start=1):
            page = futures[future]
            try:
                articles = future.result(timeout=PAGE_TIMEOUT_SECONDS)
                all_articles.extend(articles)
            except TimeoutError:
                log.error(f"Page {page + 1} exceeded the timeout.")
            except Exception as e:
                log.error(f"Error on page {page + 1}: {e}")

            if checkpoint_path and (i % 5 == 0 or i == len(pages_to_process)):
                pd.DataFrame(all_articles).to_csv(checkpoint_path, index=False)
                log.info(
                    f"Checkpoint: {i}/{len(pages_to_process)} pages processed, "
                    f"{len(all_articles)} articles so far, "
                    f"{time.time() - t0:.1f}s elapsed."
                )
    finally:
        # Threads are not spawned outside the notebook process and can
        # therefore access process_page from this notebook cell.
        executor.shutdown(wait=False, cancel_futures=True)

    return pd.DataFrame(all_articles)

def process_folder(folder_path, out_path="articles_all.csv", n_workers=1,
                    glob_pattern="*.pdf"):
    """Processes ALL PDF files in a folder and merges the results into
    a single CSV."""
    pdf_files = sorted(Path(folder_path).glob(glob_pattern))
    log.info(f"{len(pdf_files)} PDF files found in {folder_path}")

    all_dfs = []
    for pdf_file in pdf_files:
        checkpoint = f"{pdf_file.stem}_checkpoint.csv"
        log.info(f"=== Starting {pdf_file.name} ===")
        df = process_pdf(str(pdf_file), checkpoint_path=checkpoint, n_workers=n_workers)
        all_dfs.append(df)
        pd.concat(all_dfs, ignore_index=True).to_csv(out_path, index=False)

    final_df = pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame()
    final_df = final_df.drop_duplicates(subset=["source_file", "pdf_page", "text"])
    final_df.to_csv(out_path, index=False)
    log.info(f"Total: {len(final_df)} articles from {len(pdf_files)} files in {out_path}.")
    return final_df

print("Main pipeline defined.")

# 8. Sample validation

Important before interpreting the results: draws a random sample for manual review -- optionally stratified by year, since layout and OCR quality can change over the years (e.g. denser typesetting and more advertisements in later war years). A purely global sample could easily mask such period-specific issues.

In [ ]:
def validate_sample(articles_df, n=15, out_path="sample_for_review.csv",
                     stratify_by=None):
    """Draws a random sample for manual review. With
    stratify_by="year", distributed evenly across years."""
    if stratify_by and stratify_by in articles_df.columns:
        groups = articles_df.groupby(stratify_by, dropna=False)
        n_per_group = max(1, n // groups.ngroups)
        sample = groups.apply(lambda g: g.sample(min(n_per_group, len(g)))).reset_index(drop=True)
    else:
        sample = articles_df.sample(min(n, len(articles_df)))
    sample.to_csv(out_path, index=False)
    print(f"{len(sample)} articles saved to {out_path} for manual review.")
    return sample

print("Validation function defined.")



# 9. Execution

Three typical variants -- depending on what is needed. Run only the matching cell.

Before a large run: check RAM and core count so that n_workers is chosen sensibly (see the warning in Section 7):

free -h    # available RAM
nproc      # number of CPU cores

Rule of thumb: n_workers <= available RAM in GB / 1.5. It is preferable to start conservatively (e.g. 2) and increase if needed, rather than starting with too many workers and running into swapping/thrashing.

If a run takes unusually long, check in parallel in a second cell/terminal:

tail -20 extraction.log        # is progress still coming in?
ls -la *checkpoint*.csv        # when was it last updated?
free -h                        # is it swapping?


In [ ]:
import psutil

# Print total, available, and used memory in GB
mem_info = psutil.virtual_memory()
total_mem = mem_info.total / (1024 ** 3)  # GB
available_mem = mem_info.available / (1024 ** 3)  # GB
used_mem = mem_info.used / (1024 ** 3)  # GB

print(f"Total memory: {total_mem:.2f} GB")
print(f"Available memory: {available_mem:.2f} GB")
print(f"Used memory: {used_mem:.2f} GB")

In [ ]:
# Get physical and logical CPU core counts
physical_cores = psutil.cpu_count(logical=False)
logical_cores = psutil.cpu_count(logical=True)

print(f"Physical cores: {physical_cores}")
print(f"Logical cores: {logical_cores}")

# Compute the maximum reasonable number of workers based on available RAM
n_workers = int(min(logical_cores, available_mem // 1.5))  # rule of thumb: 1.5 GB/worker
print(f"Recommended number of workers: {n_workers}")


In [ ]:
# --- Variant A: test run on a few pages of a single file -------
pdf_path = "shanghaijewishch00unse.pdf"

_probe_doc = fitz.open(pdf_path)
print(f"PDF file '{pdf_path}' opened, {len(_probe_doc)} pages found.")
_n_pages_available = len(_probe_doc)
print(f"Available pages: {_n_pages_available}")
_probe_doc.close()
print(f"Test run on the first {min(4, _n_pages_available)} pages of '{pdf_path}'.")
_test_range = range(0, min(4, _n_pages_available))
print(f"Processing pages: {list(_test_range)}")

df = process_pdf(pdf_path, page_range=_test_range, n_workers=1)
df


In [ ]:
doc = None
try:
    doc = fitz.open(pdf_path)
    print(f"PDF '{pdf_path}' opened successfully.")
except Exception as e:
    print(f"Error opening the PDF file: {e}")

if doc is not None:
    _debug_page_num = min(3, len(doc) - 1)
    articles = process_page(pdf_path, page_num=_debug_page_num, n_pages=len(doc))

    try:
        pix = doc[_debug_page_num].get_pixmap(dpi=300)
        pix.save("output.png")
    except Exception as e:
        print(f"Error rendering the page: {e}")
    finally:
        doc.close()

In [ ]:
# --- Variant B: whole PDF, parallelized -----------------------------
import multiprocessing
print("Available CPU cores:", multiprocessing.cpu_count())

df = process_pdf(pdf_path, page_range=None, n_workers=3,
                   checkpoint_path="articles_checkpoint.csv")
df.to_csv("articles_extracted.csv", index=False)

In [ ]:
# --- Sample for manual review ----------------------------------
if len(df) > 0:
    sample = validate_sample(df, n=min(10, len(df)), stratify_by="year")
    sample